# KR Malaria Transmission Simulator — Example Walkthrough

This notebook walks through the full five-stage pipeline and explains how to interpret each output.

**Run from the repo root**, or adjust the paths in the setup cell below.

## 1. Run the pipeline

| Stage | Command | Output |
|---|---|---|
| 1 | `krsim simulate` | `ts_123.trees`, `sim_ind_node_map_123.csv`, `node_pairs_123.csv`, `lineage_123.csv`, `node_id_map_123.csv` |
| — | `krsim transmission-map` | `transmission_map_123.html` (needs only Stage 1's `--save_lineage`) |
| 2 | `krsim mutate` | `mutated_123.trees` |
| 3 | `krsim coal` | `coal_T_123.csv`, `mut_T_123.csv` |
| 4 | `krsim vcf-strain` | `output_123_strain.vcf` |
| 5 | `krsim vcf-host` | `output_123_host.vcf`, `host_proportions_123.csv` |

**Simulation parameters**

| Flag | Value | Meaning |
|---|---|---|
| `--Nh` | 3 | Number of human hosts |
| `--T` | 10 | Transmission generations |
| `--K` | 2 | Parasites transmitted per mosquito bite |
| `--chi` | 0.18 | Superinfection probability |
| `--C` | 1 | Oocysts per mosquito |
| `--recombination_rate` | 7.4e-7 | Per-bp recombination rate |
| `--mutation_rate` | 1e-6 | Per-bp mutation rate |
| `--random_seed` | 123 | Seed for reproducibility |
| `--save_lineage` | on | Records the per-generation pedigree needed by the extra above |

In [ ]:
import subprocess, sys, os, io
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT  = os.path.abspath('..')
OUTPUT_DIR = os.path.join(REPO_ROOT, 'examples', 'output')
SEED       = 123

os.makedirs(OUTPUT_DIR, exist_ok=True)

TS_FILE           = os.path.join(OUTPUT_DIR, f'ts_{SEED}.trees')
MUT_TS_FILE       = os.path.join(OUTPUT_DIR, f'mutated_{SEED}.trees')
STRAIN_VCF        = os.path.join(OUTPUT_DIR, f'output_{SEED}_strain.vcf')
LINEAGE_CSV       = os.path.join(OUTPUT_DIR, f'lineage_{SEED}.csv')
NODE_ID_MAP_CSV   = os.path.join(OUTPUT_DIR, f'node_id_map_{SEED}.csv')
TRANSMISSION_MAP  = os.path.join(OUTPUT_DIR, f'transmission_map_{SEED}.html')

def run(label, cmd):
    print(f'\n--- {label} ---')
    r = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
    print(r.stdout or r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'{label} failed')

run('Stage 1: simulate', [
    sys.executable, '-m', 'krsim', 'simulate',
    '--Nh', '3', '--T', '10', '--K', '2', '--chi', '0.18', '--C', '1',
    '--recombination_rate', '7.4e-7', '--random_seed', str(SEED),
    '--output_dir', OUTPUT_DIR, '--save_pairs', '--save_lineage',
])
run('Transmission map: visualize pedigree', [
    sys.executable, '-m', 'krsim', 'transmission-map',
    '--lineage_csv', LINEAGE_CSV, '--node_id_map_csv', NODE_ID_MAP_CSV,
    '--random_seed', str(SEED), '--output_dir', OUTPUT_DIR,
])
run('Stage 2: add mutations', [
    sys.executable, '-m', 'krsim', 'mutate',
    '--ts', TS_FILE, '--mutation_rate', '1e-6',
    '--random_seed', str(SEED), '--output_dir', OUTPUT_DIR,
])
run('Stage 3: coalescence & mutation times', [
    sys.executable, '-m', 'krsim', 'coal',
    '--ts', MUT_TS_FILE, '--random_seed', str(SEED), '--output_dir', OUTPUT_DIR,
])
run('Stage 4: strain VCF', [
    sys.executable, '-m', 'krsim', 'vcf-strain',
    '--ts', MUT_TS_FILE, '--random_seed', str(SEED), '--output_dir', OUTPUT_DIR,
])
run('Stage 5: host VCF + proportions', [
    sys.executable, '-m', 'krsim', 'vcf-host',
    '--strain_vcf', STRAIN_VCF,
    '--ind_node_map_csv', os.path.join(OUTPUT_DIR, f'sim_ind_node_map_{SEED}.csv'), '--random_seed', str(SEED),
    '--depth', '10', '--output_dir', OUTPUT_DIR, '--save_proportions',
])


## 2. Individual–node mapping

`sim_ind_node_map_123.csv` maps each host to its parasite strain node IDs.
A host with 2 rows carries K strains; a host with 4 rows was superinfected (2K strains).

In [ ]:
ind_node = pd.read_csv(os.path.join(OUTPUT_DIR, f'sim_ind_node_map_{SEED}.csv'))
ind_node = ind_node[ind_node['sim_id'] == SEED]
print(f"{ind_node['ind_id'].nunique()} individuals, {len(ind_node)} total nodes")
ind_node

In [ ]:
ind_node.groupby('ind_id')['node_id'].count().rename('n_strains').to_frame()

## 3. Recombinant node pairs

`node_pairs_123.csv` lists pairs of nodes produced by the same meiosis event.
Comparing their genotypes across the strain VCF reveals recombination breakpoints.

In [ ]:
pairs = pd.read_csv(os.path.join(OUTPUT_DIR, f'node_pairs_{SEED}.csv'))
pairs = pairs[pairs['sim_id'] == SEED]
print(f"{len(pairs)} recombinant pairs")
pairs

## 4. Pedigree lineage & transmission map

`lineage_123.csv` records the per-generation child→parent strain pedigree, including which
`recomb_ID` produced each child. `node_id_map_123.csv` maps those pre-simplify node IDs onto
the post-simplify IDs used everywhere else (`coal_T`, the VCFs, ...). `transmission_map_123.html`
renders the same pedigree as an interactive diagram. All three come straight off Stage 1's
`--save_lineage` output — none of them need mutations, coalescence times, or any VCF.


In [ ]:
lineage = pd.read_csv(LINEAGE_CSV)
print(f"{len(lineage)} parent-child edges across {lineage['gen'].nunique()} generations")
lineage.head(10)


In [ ]:
from IPython.display import IFrame

IFrame(src=os.path.join('output', f'transmission_map_{SEED}.html'), width=900, height=500)


## 5. Coalescence times

`coal_T_123.csv` gives the MRCA time for every pair of sample nodes, broken down by tree interval.
A higher `coal_time` means the two strains share a more ancient common ancestor at that locus.

In [ ]:
coal = pd.read_csv(os.path.join(OUTPUT_DIR, f'coal_T_{SEED}.csv'))
n_pairs = coal[['sample_1', 'sample_2']].drop_duplicates().shape[0]
print(f"{len(coal)} rows — {n_pairs} unique sample pairs across {coal['chromosome'].nunique()} chromosomes")
coal.head(10)

In [ ]:
mean_coal = coal.groupby('chromosome')['coal_time'].mean()

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(mean_coal.index, mean_coal.values)
ax.set_xlabel('Chromosome')
ax.set_ylabel('Mean coalescence time (generations)')
ax.set_title('Mean pairwise coalescence time per chromosome')
plt.tight_layout()
plt.show()

## 6. Mutation times

`mut_T_123.csv` lists every mutation placed on the genealogy — its chromosome, position, and age.
Older mutations (high `time`) are shared across more lineages; younger mutations are private to fewer strains.

In [ ]:
mut = pd.read_csv(os.path.join(OUTPUT_DIR, f'mut_T_{SEED}.csv'))
print(f"{len(mut)} mutations across {mut['chromosome'].nunique()} chromosomes")
mut.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(mut['time'], bins=30)
ax.set_xlabel('Mutation age (generations)')
ax.set_ylabel('Count')
ax.set_title('Distribution of mutation ages')
plt.tight_layout()
plt.show()

## 7. Strain-level VCF

`output_123_strain.vcf` has one sample column per parasite strain (node).
Genotypes are `0` (ref) or `1` (alt). Use `sim_ind_node_map_123.csv` to group strains by host.

In [ ]:
def read_vcf(path):
    with open(path) as f:
        lines = [l for l in f if not l.startswith('##')]
    df = pd.read_csv(io.StringIO(''.join(lines)), sep='\t')
    df.rename(columns={'#CHROM': 'CHROM'}, inplace=True)
    return df

strain_vcf = read_vcf(STRAIN_VCF)
print(f"{len(strain_vcf)} variant sites, {len(strain_vcf.columns) - 9} strain nodes")
strain_vcf.head(10)

In [ ]:
sample_cols = strain_vcf.columns[9:]
gt = strain_vcf[sample_cols].astype(int)

fig, ax = plt.subplots(figsize=(10, 4))
ax.imshow(gt.T, aspect='auto', cmap='Blues', interpolation='none')
ax.set_xlabel('Variant site')
ax.set_ylabel('Node (parasite strain)')
ax.set_yticks(range(len(sample_cols)))
ax.set_yticklabels([f'node {n}' for n in sample_cols])
ax.set_title('Genotype matrix — strain level (blue = alt allele)')
plt.tight_layout()
plt.show()

## 8. Host-level VCF

`output_123_host.vcf` aggregates strains to the host (individual) level using a **Dirichlet-Multinomial** model:

1. Strain proportions within each host are sampled once from **Dirichlet(α=1)** — all combinations equally likely.
2. **10 sequencing reads** are drawn from those proportions via Multinomial, simulating shallow sequencing of the host's blood.
3. Reads are counted by allele at each variant site:
   - `0/0` — all reads carry the ref allele
   - `1/1` — all reads carry the alt allele
   - `0/1` — reads carry both alleles (within-host mixed infection detected)

Only hosts present in the final timepoint appear as sample columns.

In [ ]:
host_vcf = read_vcf(os.path.join(OUTPUT_DIR, f'output_{SEED}_host.vcf'))
print(f"{len(host_vcf)} variant sites, {len(host_vcf.columns) - 9} individuals")
host_vcf.head(10)

In [ ]:
host_cols = host_vcf.columns[9:]
het_rate = (host_vcf[host_cols] == '0/1').mean()

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar([f'ind {c}' for c in host_cols], het_rate)
ax.set_ylabel('Proportion of sites called 0/1')
ax.set_title('Within-host heterozygosity per individual')
plt.tight_layout()
plt.show()

## 9. Host strain proportions

`host_proportions_123.csv` records the Dirichlet-sampled within-host strain proportions and Multinomial read counts
used to generate the host VCF. One row per host.

- `proportions` — fraction of the blood sample from each strain (sums to 1)
- `reads` — how many of the 10 reads came from each strain (sums to 10)

A strain with 0 reads was present but undetected at this sequencing depth.

In [ ]:
import ast

props = pd.read_csv(os.path.join(OUTPUT_DIR, f'host_proportions_{SEED}.csv'))
props['proportions'] = props['proportions'].apply(ast.literal_eval)
props['reads']       = props['reads'].apply(ast.literal_eval)
props['node_ids']    = props['node_ids'].apply(ast.literal_eval)
props

In [ ]:
fig, axes = plt.subplots(1, len(props), figsize=(4 * len(props), 3))
if len(props) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, props.iterrows()):
    labels = [f'node {n}' for n in row['node_ids']]
    ax.bar(labels, row['proportions'])
    ax.set_title(f"ind {row['ind_id']}")
    ax.set_ylabel('Dirichlet proportion')
    ax.set_ylim(0, 1)
    for tick in ax.get_xticklabels():
        tick.set_rotation(45)

fig.suptitle('Within-host strain proportions (Dirichlet sample)', y=1.02)
plt.tight_layout()
plt.show()